In [14]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18


# ── Dataset ───────────────────────────────────────────────────────────────────

# Run this once to confirm the true keypoint coordinate ranges:
def inspect_kp_ranges(root, n=50):
    xs, ys = [], []
    for file in sorted(os.listdir(root))[:n]:
        if not file.endswith("_pose.npz"):
            continue
        kp = np.load(os.path.join(root, file))["kp"][0]
        xs.append(kp[:, 0].max())
        ys.append(kp[:, 1].max())
    print(f"kp x max over {n} files: {max(xs):.1f}")
    print(f"kp y max over {n} files: {max(ys):.1f}")

# Debug showed kp x maxes at ~260, so source width is 320 (not 640).
# Uncomment to re-confirm: inspect_kp_ranges("./dataset")
KP_X_SRC = 320   # actual camera frame width your keypoints are in
KP_Y_SRC = 480   # actual camera frame height

HM_H = 256       # radar heatmap height
HM_W = 128       # radar heatmap width

VIS_THRESH = 0.3  # visibility is a float [0,1]; ignore low-confidence keypoints


class MMVRDataset(Dataset):
    def __init__(self, root, max_samples=200):
        self.samples = []
        for file in sorted(os.listdir(root)):
            if file.endswith("_radar.npz"):
                idx = file.replace("_radar.npz", "")
                self.samples.append(os.path.join(root, idx))
        self.samples = self.samples[:max_samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        base  = self.samples[i]
        radar = np.load(base + "_radar.npz")

        hori = radar["hm_hori"].astype(np.float32)   # (256, 128) — always clean
        vert = radar["hm_vert"].astype(np.float32)    # (256, 128) — NaN in this section

        # FIX 1: hm_vert is NaN across every file in this dataset section.
        # Replace with zeros so the channel exists but doesn't poison training.
        if np.isnan(vert).any():
            vert = np.zeros_like(hori)

        radar_input = np.stack([hori, vert], axis=0)          # (2, H, W)
        radar_input = radar_input / (np.max(radar_input) + 1e-6)

        pose = np.load(base + "_pose.npz")
        kp   = pose["kp"][0].copy().astype(np.float32)        # (17, 3): x, y, vis

        # FIX 2: correct source resolution — x maxes at ~260, so width = 320
        kp[:, 0] = kp[:, 0] * HM_W / KP_X_SRC   # x -> [0, 128]
        kp[:, 1] = kp[:, 1] * HM_H / KP_Y_SRC   # y -> [0, 256]

        return (
            torch.tensor(radar_input, dtype=torch.float32),
            torch.tensor(kp,          dtype=torch.float32),
        )


# ── GT heatmap generation ─────────────────────────────────────────────────────

def make_heatmaps(kp_np, H=HM_H, W=HM_W, sigma=4):
    """kp_np: (17,3)  x in [0,W), y in [0,H), vis in [0,1]"""
    maps = np.zeros((17, H, W), dtype=np.float32)
    xx, yy = np.meshgrid(np.arange(W), np.arange(H))
    for j in range(17):
        x, y, v = kp_np[j]
        if v > VIS_THRESH:
            maps[j] = np.exp(-((xx - x)**2 + (yy - y)**2) / (2 * sigma**2))
    return torch.tensor(maps, dtype=torch.float32)


# ── Model ─────────────────────────────────────────────────────────────────────

class SoftArgmax2D(nn.Module):
    def forward(self, x):
        B, C, H, W = x.shape
        x_flat = F.softmax(x.view(B, C, -1), dim=-1)
        idx    = torch.arange(H * W, device=x.device).float()
        xs     = idx % W
        ys     = torch.div(idx, W, rounding_mode='floor')
        return torch.stack([
            torch.sum(x_flat * xs, dim=-1),
            torch.sum(x_flat * ys, dim=-1),
        ], dim=-1)   # (B, C, 2)


class PoseCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(2,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
        )
        self.res = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64),
        )
        self.dec = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
        )
        self.out         = nn.Conv2d(32, 17, 1)
        self.soft_argmax = SoftArgmax2D()

    def forward(self, x):
        x        = self.enc(x)
        x        = F.relu(x + self.res(x))
        x        = self.dec(x)
        heatmaps = self.out(x)
        coords   = self.soft_argmax(heatmaps)
        return heatmaps, coords


class ResNetPose(nn.Module):
    def __init__(self):
        super().__init__()
        self.net         = resnet18(pretrained=False)
        self.net.conv1   = nn.Conv2d(2, 64, 7, stride=2, padding=3, bias=False)
        self.net.fc      = nn.Linear(512, 34)

    def forward(self, x):
        return self.net(x).view(-1, 17, 2)


# ── Training ──────────────────────────────────────────────────────────────────

dataset = MMVRDataset("./dataset", max_samples=200)
loader  = DataLoader(dataset, batch_size=8, shuffle=True, drop_last=True)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Sanity check before training
x0, kp0 = dataset[0]
assert not torch.isnan(x0).any(),  "Still NaN in radar input!"
assert not torch.isnan(kp0).any(), "NaN in keypoints!"
print(f"Input OK  — range [{x0.min():.3f}, {x0.max():.3f}]")
print(f"kp x      — range [{kp0[:,0].min():.1f}, {kp0[:,0].max():.1f}]  (expect ~[0,{HM_W}])")
print(f"kp y      — range [{kp0[:,1].min():.1f}, {kp0[:,1].max():.1f}]  (expect ~[0,{HM_H}])")

model     = PoseCNN().to(device)
baseline  = ResNetPose().to(device)
opt       = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(10):
    model.train()
    total_loss, n_ok = 0.0, 0

    for x, kp in loader:
        x  = x.to(device)
        kp = kp.to(device)

        gt_maps = torch.stack(
            [make_heatmaps(k.cpu().numpy()) for k in kp]
        ).to(device)

        pred_maps, _ = model(x)
        loss = criterion(pred_maps, gt_maps)

        if torch.isnan(loss):
            print(f"  [epoch {epoch}] NaN loss — skipping batch")
            continue

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        total_loss += loss.item()
        n_ok += 1

    print(f"Epoch {epoch:02d}  avg_loss={total_loss / max(n_ok,1):.6f}  "
          f"({n_ok}/{len(loader)} batches OK)")

# Add this right before the metrics section, after the with torch.no_grad() block
print("\n--- Raw coordinate check ---")
print("GT  (x, y) for first 5 visible kps:")
for j in range(17):
    x, y, v = kp_np[j]
    if v > VIS_THRESH:
        print(f"  kp{j:02d}: gt=({x:.1f}, {y:.1f})  cnn_pred=({pred_cnn[j,0]:.1f}, {pred_cnn[j,1]:.1f})")

# ── Metrics ───────────────────────────────────────────────────────────────────

def valid(gt, pred):
    mask = gt[:, 2] > VIS_THRESH
    return gt[mask, :2], pred[mask]

def mae(gt, pred):
    g, p = valid(gt, pred)
    return np.mean(np.linalg.norm(g - p, axis=1))

def pck(gt, pred, frac=0.05):
    g, p = valid(gt, pred)
    return np.mean(np.linalg.norm(g - p, axis=1) < frac * HM_H)

def oks(gt, pred):
    g, p = valid(gt, pred)
    d_sq = np.sum((g - p)**2, axis=1)
    return np.mean(np.exp(-d_sq / (2 * (HM_H * HM_W) * 0.05**2)))

def precision_recall_f1(gt, pred, thresh_px=5.0):
    g, p = valid(gt, pred)
    d         = np.linalg.norm(g - p, axis=1)
    tp        = np.sum(d < thresh_px)
    fp        = np.sum(d >= thresh_px)
    precision = tp / (tp + fp + 1e-6)
    recall    = tp / (len(g)  + 1e-6)
    f1        = 2 * precision * recall / (precision + recall + 1e-6)
    return precision, recall, f1


# ── Evaluation ────────────────────────────────────────────────────────────────

model.eval()
baseline.eval()

x_s, kp_s = dataset[0]
x_in = x_s.unsqueeze(0).to(device)

with torch.no_grad():
    _, pred_cnn = model(x_in)
    pred_resnet = baseline(x_in)

pred_cnn    = pred_cnn[0].cpu().numpy()
pred_resnet = pred_resnet[0].cpu().numpy()
kp_np       = kp_s.numpy()

for name, pred in [("CNN (ours)", pred_cnn), ("ResNet-18 baseline", pred_resnet)]:
    print(f"\n=== {name} ===")
    print(f"  MAE       : {mae(kp_np, pred):.3f} px")
    print(f"  PCK@5%    : {pck(kp_np, pred):.3f}")
    print(f"  OKS       : {oks(kp_np, pred):.3f}")
    p, r, f = precision_recall_f1(kp_np, pred)
    print(f"  Precision : {p:.3f}")
    print(f"  Recall    : {r:.3f}")
    print(f"  F1        : {f:.3f}")

Using device: cpu
Input OK  — range [0.000, 1.000]
kp x      — range [21.0, 92.7]  (expect ~[0,128])
kp y      — range [3.1, 252.9]  (expect ~[0,256])
Epoch 00  avg_loss=0.028255  (25/25 batches OK)
Epoch 01  avg_loss=0.002982  (25/25 batches OK)
Epoch 02  avg_loss=0.001358  (25/25 batches OK)
Epoch 03  avg_loss=0.000966  (25/25 batches OK)
Epoch 04  avg_loss=0.000823  (25/25 batches OK)
Epoch 05  avg_loss=0.000738  (25/25 batches OK)
Epoch 06  avg_loss=0.000668  (25/25 batches OK)
Epoch 07  avg_loss=0.000633  (25/25 batches OK)
Epoch 08  avg_loss=0.000607  (25/25 batches OK)
Epoch 09  avg_loss=0.000584  (25/25 batches OK)

--- Raw coordinate check ---
GT  (x, y) for first 5 visible kps:
  kp00: gt=(66.0, 17.1)  cnn_pred=(63.5, 127.4)
  kp02: gt=(64.2, 6.6)  cnn_pred=(63.5, 127.4)
  kp09: gt=(91.2, 127.1)  cnn_pred=(63.5, 127.4)

=== CNN (ours) ===
  MAE       : 86.176 px
  PCK@5%    : 0.000
  OKS       : 0.003
  Precision : 0.000
  Recall    : 0.000
  F1        : 0.000

=== ResNet-18 